<a href="https://colab.research.google.com/github/Nogueira-Amanda/Estrutura_Dados_II/blob/main/HeapSort_Sortia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!/usr/bin/env python3
"""Jogo didático de Heapsort para execução no terminal.

O estudante constrói uma max-heap e depois executa o Heapsort escolhendo
manualmente as trocas. O programa valida cada decisão e explica os índices.
Não usa bibliotecas externas.
"""

from __future__ import annotations

import random
from dataclasses import dataclass

# Armazena as informações do desempenho do jogador durante a atividade.
@dataclass
class Placar:
    acertos: int = 0
    erros: int = 0
    dicas: int = 0

    @property
    def pontos(self) -> int:                                                # Cada acerto vale 10 pontos. Erros e dicas reduzem a pontuação.
        return max(0, self.acertos * 10 - self.erros * 2 - self.dicas)      # max() impede que o resultado final seja negativo.


def filhos(indice: int, tamanho: int) -> list[int]:
    """Retorna os índices dos filhos existentes de um nó."""
    candidatos = (2 * indice + 1, 2 * indice + 2)      # Em uma heap representada por vetor, os filhos de um nó de índice i são:
    return [i for i in candidatos if i < tamanho]      # filho esquerdo = 2*i + 1 e filho direito = 2*i + 2.
                                                       # Retorna somente os filhos que realmente existem dentro da heap ativa.


def maior_filho(vetor: list[int], indice: int, tamanho: int) -> int | None:
    """Retorna o índice do maior filho, priorizando o esquerdo em empate."""
    indices = filhos(indice, tamanho)                                   # Se houver filhos, procura aquele cujo valor no vetor é o maior.
    return max(indices, key=lambda i: vetor[i]) if indices else None    # Caso o nó não tenha filhos, retorna None.


def tem_propriedade_heap(vetor: list[int], indice: int, tamanho: int) -> bool:
    """Verifica a propriedade de max-heap em um nó."""                  # Em uma max-heap, o pai deve possuir valor maior ou igual ao valor de cada um de seus filhos.
    return all(vetor[indice] >= vetor[f] for f in filhos(indice, tamanho))


def eh_max_heap(vetor: list[int], tamanho: int) -> bool:
    """Verifica se o prefixo vetor[:tamanho] é uma max-heap."""         # Apenas os nós da primeira metade precisam ser verificados, pois os elementos restantes são folhas e não possuem filhos.
    return all(tem_propriedade_heap(vetor, i, tamanho) for i in range(tamanho // 2))


def exibir_arvore(vetor: list[int], tamanho_heap: int) -> None:
    """Mostra a heap por níveis e separa a região já ordenada."""
    print("\nVetor:", " ".join(f"[{i}:{v}]" for i, v in enumerate(vetor)))           # Exibe cada elemento no formato [índice:valor].
    print("Heap ativa:", vetor[:tamanho_heap], "| Ordenados:", vetor[tamanho_heap:]) # Durante o Heapsort, o vetor é dividido em duas regiões:
    if tamanho_heap == 0:                                                            # heap ativa e elementos que já estão em suas posições definitivas.
        return                                                                       # Se não existir mais heap ativa, não há árvore para exibir.
    print("Árvore lógica (índice:valor):")                                           # A heap é armazenada em vetor, mas pode ser visualizada como árvore.
    inicio, largura = 0, 1                                                           # Cada nível possui, no máximo, o dobro de nós do nível anterior.
    while inicio < tamanho_heap:
        fim = min(inicio + largura, tamanho_heap)
        print("  " + "   ".join(f"{i}:{vetor[i]}" for i in range(inicio, fim)))
        inicio, largura = fim, largura * 2                                           # Avança para o próximo nível da árvore.


def ler_escolha(prompt: str, opcoes: set[str]) -> str:
    while True:
        resposta = input(prompt).strip().lower()
        if resposta in opcoes:                                       # A função só termina quando o usuário informa uma opção permitida.
            return resposta
        print("Opção inválida. Escolha:", ", ".join(sorted(opcoes)))


def perguntar_troca(
    vetor: list[int], indice_pai: int, indice_filho: int, placar: Placar
) -> None:
    """Solicita a troca correta entre pai e maior filho."""
    while True:
        print(f"O nó {indice_pai}:{vetor[indice_pai]} viola a propriedade de max-heap.")
        resposta = input(
            "Digite os índices que devem ser trocados (ex.: 1 4) ou 'dica': "
        ).strip().lower()
        if resposta == "dica":    # O jogador pode solicitar ajuda sem realizar uma tentativa de troca.
            placar.dicas += 1
            print(
                "Dica: compare os dois filhos e troque o pai com o filho de maior valor."
            )
            continue
        try:                     # Converte os dois índices digitados para números inteiros.
            a, b = map(int, resposta.split())
        except ValueError:
            print("Informe exatamente dois índices inteiros.")
            continue
        if {a, b} == {indice_pai, indice_filho}:      # O uso de conjuntos permite aceitar os índices em qualquer ordem. Ex.: "1 4" e "4 1" representam a mesma troca.
            vetor[indice_pai], vetor[indice_filho] = vetor[indice_filho], vetor[indice_pai]   # Troca os valores das duas posições diretamente no vetor.
            placar.acertos += 1
            print("✓ Troca correta!")
            return
        placar.erros += 1
        print("✗ Essa troca não restaura a propriedade de heap neste nó. Tente novamente.")


def descer_manual(vetor: list[int], raiz: int, tamanho: int, placar: Placar) -> None:
    """Restaura a heap, exigindo que o estudante escolha cada troca."""
    pai = raiz      # Começa pelo nó indicado como raiz da subárvore analisada.
    while True:     # Localiza o filho de maior valor.
        filho = maior_filho(vetor, pai, tamanho)
        if filho is None or vetor[pai] >= vetor[filho]:   # A descida termina se o nó não possuir filhos ou se o pai já for maior ou igual ao maior filho.
            print(f"O nó {pai}:{vetor[pai]} já possui a propriedade de heap.")
            return
        exibir_arvore(vetor, tamanho)               # Se o pai for menor que o maior filho, a propriedade da max-heap foi violada e uma troca será necessária.
        perguntar_troca(vetor, pai, filho, placar)  # Depois da troca, continua verificando a posição para onde o antigo valor do pai desceu.
        pai = filho


def construir_heap(vetor: list[int], placar: Placar) -> None:
    print("\n=== FASE 1 — CONSTRUIR A MAX-HEAP ===")    # O último nó que pode possuir filhos está em len(vetor)//2 - 1.
    print("Percorreremos os pais de baixo para cima.")  # A construção começa nele e segue até a raiz, índice 0.
    for pai in range(len(vetor) // 2 - 1, -1, -1):
        print(f"\nAnalisando o nó de índice {pai}...")
        descer_manual(vetor, pai, len(vetor), placar)
    assert eh_max_heap(vetor, len(vetor))               # Confirma internamente que a construção foi realizada corretamente.
    exibir_arvore(vetor, len(vetor))
    print("✓ Max-heap construída: o maior valor está na raiz.")


def ordenar_manual(vetor: list[int], placar: Placar) -> None:
    print("\n=== FASE 2 — ORDENAR ===")
    for fim in range(len(vetor) - 1, 0, -1):    # 'fim' representa a última posição pertencente à heap ativa.
        exibir_arvore(vetor, fim + 1)           # A cada repetição, essa região diminui uma posição.
        print("A raiz contém o maior valor da heap ativa.")
        while True:
            resposta = input(
                f"Quais índices devem ser trocados para fixar o máximo na posição {fim}? "
            ).strip().lower()
            if resposta == "dica":
                placar.dicas += 1
                print(f"Dica: troque a raiz (índice 0) com o último índice ativo ({fim}).")
                continue
            try:
                a, b = map(int, resposta.split())
            except ValueError:
                print("Digite dois índices ou 'dica'.")
                continue
            if {a, b} == {0, fim}:     # Como o maior elemento está na raiz da max-heap, ele deve ser trocado com o último elemento da região ativa.
                vetor[0], vetor[fim] = vetor[fim], vetor[0]
                placar.acertos += 1
                print("✓ Máximo colocado em sua posição definitiva!")
                break
            placar.erros += 1
            print("✗ A troca deve retirar a raiz e reduzir a heap ativa.")    # Após retirar o maior elemento, a raiz pode deixar de respeitar
        descer_manual(vetor, 0, fim, placar)                                  # a propriedade da heap. Por isso, ela é reorganizada novamente.
                                                                              # O índice 'fim' não participa mais, pois já está ordenado.

def obter_vetor() -> list[int]:
    print("\n1 — Usar números aleatórios")
    print("2 — Digitar os números")
    escolha = ler_escolha("Escolha: ", {"1", "2"})
    if escolha == "1":   # Gera sete valores diferentes entre 1 e 99.
        return random.sample(range(1, 100), 7)
    while True:
        try:
            valores = [int(x) for x in input("Digite de 4 a 12 inteiros separados por espaço: ").split()] # split() separa a entrada pelos espaços e int() converte cada parte digitada para um número inteiro.
            if 4 <= len(valores) <= 12:                                                                   # O jogo aceita vetores contendo entre 4 e 12 elementos.
                return valores
        except ValueError:   # O erro ocorre se algum dos valores não puder ser convertido para um número inteiro.
            pass
        print("Entrada inválida. Use somente inteiros e informe entre 4 e 12 valores.")


def main() -> None:
    print("=" * 58)
    print("JOGO DE HEAPSORT — APRENDER FAZENDO")
    print("=" * 58)
    print("Em um vetor com índice i: esquerdo = 2i+1; direito = 2i+2.")      # Apresenta ao jogador as duas regras fundamentais utilizadas para representar e organizar uma max-heap.
    print("Regra da max-heap: todo pai deve ser maior ou igual aos filhos.")
    vetor = obter_vetor()          # Obtém o vetor que será utilizado durante a partida.
    original = vetor.copy()        # copy() é necessário porque o Heapsort modifica o próprio vetor.
    placar = Placar()              # Cria o placar inicialmente com zero acertos, erros e dicas.
    construir_heap(vetor, placar)  # Fase 1: transforma o vetor em uma max-heap.
    ordenar_manual(vetor, placar)  # Fase 2: utiliza a max-heap para ordenar os elementos.
    exibir_arvore(vetor, 0)        # tamanho_heap = 0 indica que todos os elementos estão ordenados.
    assert vetor == sorted(original)          # Verificação interna: compara o resultado do jogo com a ordenação produzida pela função sorted() do Python.
    print("\nParabéns! Ordenação concluída.")
    print("Original:", original)
    print("Ordenado:", vetor)
    print(
        f"Placar: {placar.pontos} pontos "    # Apresenta a pontuação e o desempenho final do estudante.
        f"({placar.acertos} acertos, {placar.erros} erros, {placar.dicas} dicas)"
    )

# Esta condição garante que main() seja executada somente quando este arquivo for iniciado diretamente, e não quando for importado.
if __name__ == "__main__":
    main()


JOGO DE HEAPSORT — APRENDER FAZENDO
Em um vetor com índice i: esquerdo = 2i+1; direito = 2i+2.
Regra da max-heap: todo pai deve ser maior ou igual aos filhos.

1 — Usar números aleatórios
2 — Digitar os números
Escolha: 2
Digite de 4 a 12 inteiros separados por espaço: 88 59 26 86 70 33 32

=== FASE 1 — CONSTRUIR A MAX-HEAP ===
Percorreremos os pais de baixo para cima.

Analisando o nó de índice 2...

Vetor: [0:88] [1:59] [2:26] [3:86] [4:70] [5:33] [6:32]
Heap ativa: [88, 59, 26, 86, 70, 33, 32] | Ordenados: []
Árvore lógica (índice:valor):
  0:88
  1:59   2:26
  3:86   4:70   5:33   6:32
O nó 2:26 viola a propriedade de max-heap.
Digite os índices que devem ser trocados (ex.: 1 4) ou 'dica': 5 2
✓ Troca correta!
O nó 5:26 já possui a propriedade de heap.

Analisando o nó de índice 1...

Vetor: [0:88] [1:59] [2:33] [3:86] [4:70] [5:26] [6:32]
Heap ativa: [88, 59, 33, 86, 70, 26, 32] | Ordenados: []
Árvore lógica (índice:valor):
  0:88
  1:59   2:33
  3:86   4:70   5:26   6:32
O nó 1:5